# `felix_02_genome_review` — review a FelixGenome run

Reads one FelixGenome run and produces every plot and table needed to decide
whether it worked, for **both** arms (ordinary per-record scan and repeat-dosage
scan) plus the biallelic comparator when it is present.

Run **on the Terra Workbench VM**.

## What it does

1. Pulls `felix_run.summary.tar.gz` (workflow output `run_summary_tar`) — a few
   hundred kB holding the manifests, calibration tables, QQ/Manhattan plots and
   the per-chromosome aggregator/writer stats.
2. Builds a combined calibration table and a λGC figure across arms.
3. Checks the repeat-branch QC fields per chromosome against their thresholds.
4. Displays the per-phenotype QQ and Manhattan plots.
5. Compares the dosage and biallelic arms where both ran.
6. Looks up named loci (expansion positive controls) by streaming only the
   matching rows out of the large per-phenotype TSVs, which stay in the bucket.

The full per-phenotype scans are **not** downloaded. Genome-wide they are tens
of GB per phenotype; step 6 streams them through `awk` and keeps only matching
rows.

## Config

In [ ]:
import os
import subprocess
from pathlib import Path

# Either set FELIX_RUN_ROOT directly, or the submission/workflow ids.
RUN_ROOT = os.environ.get("FELIX_RUN_ROOT", "").rstrip("/")
if not RUN_ROOT:
    bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    sub = os.environ.get("FELIX_SUBMISSION_ID", "")
    wf = os.environ.get("FELIX_WORKFLOW_ID", "")
    if bucket and sub and wf:
        RUN_ROOT = f"{bucket}/submissions/{sub}/FelixGenome/{wf}"
if not RUN_ROOT.startswith("gs://"):
    raise ValueError(
        "set FELIX_RUN_ROOT=gs://.../FelixGenome/<workflow-id>, or "
        "FELIX_SUBMISSION_ID and FELIX_WORKFLOW_ID with WORKSPACE_BUCKET"
    )

WORK = Path(os.environ.get("FELIX_REVIEW_DIR", "felix_review")).resolve()
WORK.mkdir(parents=True, exist_ok=True)

# Phenotypes in selected_phenotypes order; Concat/RuConcat shards follow it.
PHENOTYPES = [p for p in os.environ.get("FELIX_PHENOTYPES", "").split(",") if p]

LAMBDA_OK = (0.95, 1.05)          # per-phenotype lambda_gc_cct band
MIN_COVERAGE_RATE = 0.95          # not used here; propagate's own threshold
MAX_ANC_INCONSISTENT = 0.01       # hap_ancestry_inconsistent_frac
MAX_POSSIBLE_DUPLICATE = 0.02     # hap_possible_duplicate_frac

print("run root:", RUN_ROOT)
print("work dir:", WORK)


def gs(*args, text=True, check=True):
    """Run gsutil and return stdout."""
    return subprocess.run(["gsutil", *args], check=check, capture_output=True, text=text).stdout


def gs_ls(pattern):
    try:
        out = gs("ls", pattern)
    except (subprocess.CalledProcessError, FileNotFoundError):
        return []
    return [u for u in out.split() if u.startswith("gs://")]


## 1. Fetch the run summary bundle

`BundleRunSummary` writes one tarball per run. Older runs (before that task
existed) have no bundle, so the fallback assembles the same tree from the
individual task outputs.

In [ ]:
import tarfile

BUNDLE = WORK / "bundle"
if BUNDLE.exists():
    print("reusing", BUNDLE)
else:
    tars = gs_ls(f"{RUN_ROOT}/**/felix_run.summary.tar.gz")
    BUNDLE.mkdir(parents=True, exist_ok=True)
    if tars:
        local = WORK / "felix_run.summary.tar.gz"
        gs("cp", tars[0], str(local))
        with tarfile.open(local) as tf:
            tf.extractall(BUNDLE)
        print("unpacked", tars[0])
    else:
        print("no felix_run.summary.tar.gz under the run root; assembling from task outputs")
        # arm summaries: call-Summarize (main), call-SummarizeRu, call-SummarizeBa
        for call, arm in (("Summarize", "main"), ("SummarizeRu", "ru"), ("SummarizeBa", "ru_biallelic")):
            got = False
            for name in ("calibration_summary.tsv", "results_manifest.tsv"):
                uris = gs_ls(f"{RUN_ROOT}/call-{call}/**/summary/{name}")
                if uris:
                    (BUNDLE / arm).mkdir(parents=True, exist_ok=True)
                    gs("cp", uris[0], str(BUNDLE / arm / name))
                    got = True
            if got:
                qc = gs_ls(f"{RUN_ROOT}/call-{call}/**/summary/qc/*/*")
                for u in qc:
                    pheno = u.rsplit("/", 2)[-2]
                    d = BUNDLE / arm / "qc" / pheno
                    d.mkdir(parents=True, exist_ok=True)
                    gs("cp", u, str(d / u.rsplit("/", 1)[-1]))
        # per-chromosome repeat QC
        rq = BUNDLE / "repeat_qc"
        # WriteRuAdmixedVcf writes ru.admixed.stats.json for both arms, so the
        # remote name is shared and only the local tag distinguishes them.
        for call, remote, tag in (("RuLoci", "loci.summary.json", "loci.summary.json"),
                                  ("RuVcf", "ru.admixed.stats.json", "ru.admixed.stats.json"),
                                  ("BaVcf", "ru.admixed.stats.json", "ru_biallelic.admixed.stats.json")):
            uris = gs_ls(f"{RUN_ROOT}/call-{call}/shard-*/**/{remote}")
            for u in uris:
                shard = next((p for p in u.split("/") if p.startswith("shard-")), "shard-?")
                rq.mkdir(parents=True, exist_ok=True)
                gs("cp", u, str(rq / f"{shard}.{tag}"))

for p in sorted(BUNDLE.rglob("*")):
    if p.is_file():
        print(p.relative_to(BUNDLE))


## 2. Calibration across arms

`lambda_gc_cct` is the genomic-control inflation factor on the Cauchy-combined
joint p-value. The arms share one GRM and one null fit per phenotype, so their
λGC values are directly comparable: a repeat arm far from the main arm points at
the encoding, not at the phenotype.

In [ ]:
import pandas as pd

ARMS = ["main", "ru", "ru_biallelic"]
ARM_LABEL = {"main": "per-record scan", "ru": "repeat dosage", "ru_biallelic": "repeat biallelic"}

frames = []
for arm in ARMS:
    f = BUNDLE / arm / "calibration_summary.tsv"
    if f.is_file():
        df = pd.read_csv(f, sep="\t")
        df.insert(0, "arm", arm)
        frames.append(df)
if not frames:
    raise FileNotFoundError(f"no calibration_summary.tsv under {BUNDLE}")
cal = pd.concat(frames, ignore_index=True)
cal.to_csv(WORK / "calibration_all_arms.tsv", sep="\t", index=False)

wide = cal.pivot(index="phenotype", columns="arm", values="lambda_gc_cct")
nvar = cal.pivot(index="phenotype", columns="arm", values="n_variants")
if PHENOTYPES:
    wide = wide.reindex([p for p in PHENOTYPES if p in wide.index])
    nvar = nvar.reindex(wide.index)

print("lambda_gc_cct by arm")
display(wide.round(4))
print("\nvariants / loci tested by arm")
display(nvar)

out_of_band = wide[(wide < LAMBDA_OK[0]) | (wide > LAMBDA_OK[1])].stack().dropna()
print(f"\nphenotype x arm outside {LAMBDA_OK}:", len(out_of_band))
if len(out_of_band):
    display(out_of_band.round(4))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

present = [a for a in ARMS if a in wide.columns]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})

ax = axes[0]
x = np.arange(len(wide.index))
w = 0.8 / max(len(present), 1)
for i, arm in enumerate(present):
    ax.bar(x + i * w - 0.4 + w / 2, wide[arm].to_numpy(), width=w, label=ARM_LABEL[arm])
ax.axhline(1.0, color="0.3", lw=1)
ax.axhspan(*LAMBDA_OK, color="0.85", zorder=0)
ax.set_xticks(x); ax.set_xticklabels(wide.index, rotation=45, ha="right")
ax.set_ylabel(r"$\lambda_{GC}$ (P_cct)"); ax.set_ylim(min(0.9, wide.min().min() - 0.02), max(1.1, wide.max().max() + 0.02))
ax.legend(frameon=False, fontsize=8); ax.set_title("Calibration by phenotype and arm")

ax = axes[1]
if "ru" in wide.columns and "main" in wide.columns:
    ax.scatter(wide["main"], wide["ru"], s=28, color="#1f4e79")
    lim = [min(wide[["main", "ru"]].min()) - 0.02, max(wide[["main", "ru"]].max()) + 0.02]
    ax.plot(lim, lim, color="0.5", lw=1, ls="--"); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(r"per-record scan $\lambda_{GC}$"); ax.set_ylabel(r"repeat dosage $\lambda_{GC}$")
    ax.set_title("Repeat vs per-record, same null")
else:
    ax.set_axis_off(); ax.text(0.5, 0.5, "repeat arm not present", ha="center")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(WORK / "calibration_by_arm.png", dpi=200)
plt.show()


## 3. Repeat-branch QC per chromosome

Read these before any p-value. `anc_code_max` must equal `num_ancs - 1`, and a
`written` count far below `loci_polymorphic` means loci were dropped between the
aggregator and the dosage VCF.

In [ ]:
import json
import re

rows = []
for f in sorted((BUNDLE / "repeat_qc").glob("*loci.summary.json")) if (BUNDLE / "repeat_qc").is_dir() else []:
    d = json.loads(f.read_text())
    c = d.get("counts", {})
    key = f.name.split(".")[0]
    rows.append({
        "chrom": d.get("region", key), "n_samples": d.get("n_samples"),
        "loci_in_catalog": c.get("loci_in_catalog"), "loci_with_records": c.get("loci_with_records"),
        "loci_polymorphic": c.get("loci_polymorphic"), "records_read": c.get("records_read"),
        "records_unassigned": c.get("records_unassigned"), "anc_code_max": d.get("anc_code_max"),
        "hap_ancestry_inconsistent_frac": d.get("hap_ancestry_inconsistent_frac"),
        "hap_possible_duplicate_frac": d.get("hap_possible_duplicate_frac"),
        "hap_missing_frac": d.get("hap_missing_frac"),
        "hap_unphased_ambiguous": c.get("hap_unphased_ambiguous"),
    })
loci_qc = pd.DataFrame(rows)

rows = []
for f in sorted((BUNDLE / "repeat_qc").glob("*ru.admixed.stats.json")) if (BUNDLE / "repeat_qc").is_dir() else []:
    d = json.loads(f.read_text())
    d["chrom"] = f.name.split(".")[0]
    rows.append(d)
vcf_qc = pd.DataFrame(rows)

if len(loci_qc):
    qc = loci_qc.merge(vcf_qc[["chrom", "records_in", "written", "monomorphic_skipped",
                               "missingness_skipped", "coding"]], on="chrom", how="left") \
        if len(vcf_qc) else loci_qc
    qc.to_csv(WORK / "repeat_qc_by_chrom.tsv", sep="\t", index=False)
    display(qc)

    flags = []
    anc = qc.anc_code_max.dropna().unique()
    if len(anc) != 1:
        flags.append(f"anc_code_max differs across chromosomes: {sorted(anc)}")
    if "written" in qc and (qc.written.fillna(-1) != qc.loci_polymorphic).any():
        bad = qc.loc[qc.written.fillna(-1) != qc.loci_polymorphic, "chrom"].tolist()
        flags.append(f"written != loci_polymorphic on {bad}")
    for col, cap in (("hap_ancestry_inconsistent_frac", MAX_ANC_INCONSISTENT),
                     ("hap_possible_duplicate_frac", MAX_POSSIBLE_DUPLICATE)):
        over = qc.loc[qc[col].fillna(0) > cap, "chrom"].tolist()
        if over:
            flags.append(f"{col} > {cap} on {over}")
    print("\nFLAGS:" if flags else "\nno QC flags raised")
    for f_ in flags:
        print(" -", f_)
    print("\nanc_code_max =", sorted(anc), "(must be num_ancs - 1)")
else:
    print("no repeat QC JSONs in the bundle")


## 4. QQ and Manhattan plots

In [ ]:
from IPython.display import Image, display as disp

def show_arm(arm, kind="qq_cct.png", phenos=None):
    d = BUNDLE / arm / "qc"
    if not d.is_dir():
        print(f"{arm}: no qc/ in bundle"); return
    names = phenos or sorted(p.name for p in d.iterdir() if p.is_dir())
    for p in names:
        f = d / p / kind
        if f.is_file():
            print(f"{ARM_LABEL.get(arm, arm)} — {p} — {kind}")
            disp(Image(filename=str(f)))

SHOW = PHENOTYPES[:3] if PHENOTYPES else None   # set to None for every phenotype
for arm in [a for a in ARMS if (BUNDLE / a).is_dir()]:
    show_arm(arm, "qq_cct.png", SHOW)


In [ ]:
for arm in [a for a in ARMS if (BUNDLE / a).is_dir()]:
    show_arm(arm, "manhattan_cct.png", SHOW)


## 5. Dosage vs biallelic arm

`CompareRuArms` writes one table per phenotype when the biallelic comparator arm
ran. The dosage arm wins where a multi-allelic repeat's signal is spread across
alleles by the biallelic coding.

In [ ]:
arm_dir = BUNDLE / "arm_compare"
if arm_dir.is_dir() and any(arm_dir.iterdir()):
    summaries = sorted(arm_dir.glob("*summary*"))
    tables = sorted(p for p in arm_dir.glob("*") if p not in summaries and p.suffix in (".tsv", ".csv"))
    print("arm_compare files:", [p.name for p in sorted(arm_dir.iterdir())][:20])
    frames = []
    for p in summaries:
        try:
            df = pd.read_json(p, typ="series").to_frame().T if p.suffix == ".json" else pd.read_csv(p, sep="\t")
        except ValueError:
            continue
        df.insert(0, "file", p.name)
        frames.append(df)
    if frames:
        display(pd.concat(frames, ignore_index=True))
else:
    print("no arm_compare outputs; the biallelic comparator arm did not run")


## 6. Named loci (expansion positive controls)

The per-phenotype scans stay in the bucket; this streams them through `awk` and
keeps only rows inside the requested windows.

**The coordinates below are hg38 and are NOT verified in this notebook.** Each
entry carries the motif the locus is expected to have, and the lookup reports the
motif it actually matched, so a wrong window shows up as a missing or
unexpected-motif hit rather than a silently wrong answer. Check any control you
intend to report.

In [ ]:
# name -> (chrom, start, end, expected motif). Windows are padded below.
POSITIVE_CONTROLS = {
    "HTT":      ("chr4",  3074877,  3074940, "CAG"),
    "C9orf72":  ("chr9", 27573484, 27573546, "GGGGCC"),
    "FXN":      ("chr9", 69037286, 69037304, "GAA"),
    "RFC1":     ("chr4", 39348425, 39348479, "AAGGG"),
    "ATXN3":    ("chr14", 92071010, 92071042, "CTG"),
    "DMPK":     ("chr19", 45770205, 45770264, "CAG"),
}
PAD = 5000          # bp either side, because the catalog interval may differ
VERIFIED = False    # set True only after checking against your own source

windows = {k: (c, max(1, s - PAD), e + PAD, m) for k, (c, s, e, m) in POSITIVE_CONTROLS.items()}
for k, (c, s, e, m) in windows.items():
    print(f"{k:9s} {c}:{s}-{e}  expect motif {m}")
if not VERIFIED:
    print("\nVERIFIED=False — treat hits as candidates until the coordinates are checked")


In [ ]:
def extract_windows(uri, windows, out_path):
    """Stream a results TSV from GCS, keeping the header plus rows inside any window."""
    spec = "\n".join(f"{c}\t{s}\t{e}\t{name}" for name, (c, s, e, _m) in windows.items())
    Path("windows.tmp").write_text(spec + "\n")
    awk = r"""
      NR==FNR { n++; chr[n]=$1; lo[n]=$2; hi[n]=$3; nm[n]=$4; next }
      FNR==1  { print "locus_name	" $0; next }
      { for (i = 1; i <= n; i++)
          if ($1 == chr[i] && $2 >= lo[i] && $2 <= hi[i]) { print nm[i] "	" $0; break } }
    """
    with open(out_path, "w") as fh:
        p1 = subprocess.Popen(["gsutil", "cat", uri], stdout=subprocess.PIPE)
        p2 = subprocess.Popen(["awk", "-F", "\t", "-v", "OFS=\t", awk, "windows.tmp", "-"],
                              stdin=p1.stdout, stdout=fh)
        p1.stdout.close()
        p2.communicate()
    return out_path


def arm_result_uris(call):
    """Per-phenotype merged TSVs for a Concat-style call, in shard order."""
    uris = gs_ls(f"{RUN_ROOT}/call-{call}/shard-*/**/merged.felix.tsv")
    def shard_of(u):
        m = re.search(r"/shard-(\d+)/", u)
        return int(m.group(1)) if m else -1
    return [u for _, u in sorted((shard_of(u), u) for u in uris)]


ru_uris = arm_result_uris("RuConcat")
print(f"{len(ru_uris)} repeat-arm result files")
hits = []
for i, uri in enumerate(ru_uris):
    pheno = PHENOTYPES[i] if i < len(PHENOTYPES) else f"shard{i}"
    out = WORK / f"controls_ru_{pheno}.tsv"
    if not out.is_file():
        extract_windows(uri, windows, out)
    df = pd.read_csv(out, sep="\t", low_memory=False)
    if len(df):
        df.insert(0, "phenotype", pheno)
        hits.append(df)
    print(f"  {pheno}: {len(df)} rows")

controls = pd.concat(hits, ignore_index=True) if hits else pd.DataFrame()
if len(controls):
    controls.to_csv(WORK / "positive_controls_ru.tsv", sep="\t", index=False)
    keep = ["phenotype", "locus_name", "CHR", "POS", "MarkerID", "P_cct_admixed_c", "P_hom_admixed_c"]
    keep = [c for c in keep if c in controls.columns]
    display(controls[keep].sort_values(["locus_name", "P_cct_admixed_c"]).head(40))
    print("\nloci found per control:")
    display(controls.groupby("locus_name").MarkerID.nunique())
else:
    print("no rows matched any control window")


## 7. Review note

A short markdown file recording what was checked, to keep with the run.

In [ ]:
def block(df, title):
    """Fenced plain-text table: to_markdown() needs tabulate, which the VM may lack."""
    body = df.to_string() if hasattr(df, "to_string") else str(df)
    return [f"## {title}", "", "```", body, "```", ""]

lines = ["# FelixGenome run review", "", f"Run root: `{RUN_ROOT}`", ""]
lines += block(wide.round(4), "Calibration (lambda_gc_cct)")
if "qc" in dir() and len(qc):
    lines += block(qc, "Repeat QC by chromosome")
if len(out_of_band):
    lines += block(out_of_band.round(4), f"Phenotype x arm outside {LAMBDA_OK}")
else:
    lines += [f"All phenotype x arm lambda values inside {LAMBDA_OK}.", ""]
if "controls" in dir() and len(controls):
    lines += block(controls.groupby("locus_name").MarkerID.nunique(),
                   f"Positive-control loci (VERIFIED={VERIFIED}, window pad {PAD} bp)")
note = WORK / "run_review.md"
note.write_text("\n".join(lines) + "\n")
print(note)
print("\n".join(lines))
